In [1]:

import os
import scipy.io
import h5py
import numpy as np
import sys
from scipy import signal as sig
from scipy import stats as stats
import math
import pandas as pd
import csv
import matplotlib.pyplot as plt
from scipy import signal
import seaborn as sns
import logging

%matplotlib widget

scripts_path = '/home/apolo/Documents/github_projects/ripple_sync/scripts/hc_11_scripts/'
os.chdir(scripts_path)

import hc11_BaseFunctions as hc11_bf

scripts_path = '/home/apolo/Documents/github_projects/ripple_sync/scripts/hc_18_scripts/'
os.chdir(scripts_path)

import hc18_BaseFunctions as hc18_bf

sessions = ['Train-242-20140124','Train-261-20140617','Train-272-20141215','Train-292-20150501','Train-314-20160118']
animals = ['242','261','272','292','314']


In [2]:

# event_name = '-SleepBaseline'
event_name = '-PassiveForward'

session = sessions[4]

lfp,srate = hc18_bf.load_lfp_hc18_data(session)
    
event_times, event_descriptions = hc18_bf.get_events(session, event_name)
    
event_times = np.sort(np.array(event_times))
# event_start = np.nanmin(event_times)
# event_end = np.nanmax(event_times)

event_start = event_times[0]
event_end = event_times[1]
shanks_electrodes,left_shanks,right_shanks = hc18_bf.get_shanks_info(session)

lfp_index = np.arange(np.round(event_start*srate),np.round(event_end*srate)).astype(int)

lfp = np.double(lfp[:,lfp_index])
lfp = sig.detrend(hc11_bf.eegfilt(lfp,srate,1,0))


In [3]:
session

'Train-314-20160118'

In [5]:
window = 2*srate
f, pxx = sig.welch(lfp, fs=srate, window='hann', nperseg=window, noverlap=window/2, nfft=2**14, detrend='constant', return_onesided=True, scaling='density', axis=-1, average='mean')


In [25]:
channels = shanks_electrodes[16]

color_vec = ['black','magenta','blue','darkgreen','yellow','lightgreen','brown','red']
plt.figure(figsize=(18,6))

for counter,channel in enumerate(channels):
        
    plt.plot(f, pxx[channel,:],ls = '-', color = color_vec[counter],label = channel)
plt.legend()
plt.xlim([0,60])
plt.xlabel('Frequency [Hz]')
plt.ylabel('Power')
plt.show()




IndexError: list index out of range

In [193]:
session

'Train-292-20150501'

'Train-261-20140617'

In [ ]:

max_ripple_channels = []
for channels in shanks_electrodes:
    ripple_lfp = lfp[channels,:][lfp_index]
    ripple_filtered = sig.detrend(hc11_bf.eegfilt(ripple_lfp,srate,100,250))
    ripple_hilbert = hc11_bf.hilbert(ripple_filtered)
    ripple_amp = np.abs(ripple_hilbert)

    max_ripple_idx = np.argmax(np.nanmean(ripple_amp,1))
    max_ripple_channels.append(channels[max_ripple_idx])
max_ripple_channels = np.array(max_ripple_channels)
